In [ ]:
# Demo (PyTorch)

# Overview (Training Utilities in PyTorch)
- **ModelCheckpoint**: lưu trọng số tốt nhất theo `val_loss` (`.pt`)
- **EarlyStopping**: dừng huấn luyện sớm để tránh overfitting (`patience=5`)
- **ReduceLROnPlateau**: giảm learning rate khi `val_loss` không cải thiện (`factor=0.1, patience=5, min_lr=0.0001`)
- **TensorBoard**: ghi log loss & accuracy bằng `torch.utils.tensorboard.SummaryWriter`

# Import libraries

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

try:
    from torch.utils.tensorboard import SummaryWriter
    HAS_TENSORBOARD = True
except ImportError:
    HAS_TENSORBOARD = False
    print('TensorBoard chưa được cài đặt. Chạy `pip install tensorboard` nếu muốn ghi log.')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
num_classes = 2   # male vs. female

# (1) Data

### Download data at https://drive.google.com/drive/folders/0BxINLo5jshCRYW8xODhNSlkyLTQ?resourcekey=0-RNJvvSSE67Q1bMqQi3A1RQ&usp=sharing
Dữ liệu `.npy` nằm trong `data/gender/` — chạy notebook từ thư mục gốc BTL03

In [ ]:
x_train = np.load('data/gender/64_64_11938_4098_train_x_onehot.npy')
y_train_oh = np.load('data/gender/64_64_11938_4098_train_y_onehot.npy')
x_train = np.rot90(x_train, k=3, axes=(1, 2)).copy()

x_test = np.load('data/gender/64_64_5968_4098_val_x_onehot.npy')
y_test_oh = np.load('data/gender/64_64_5968_4098_val_y_onehot.npy')
x_test = np.rot90(x_test, k=3, axes=(1, 2)).copy()

# Chuyển NHWC (N, 64, 64, 1) -> NCHW (N, 1, 64, 64) cho PyTorch Conv2d
if x_train.ndim == 3:
    x_train = np.expand_dims(x_train, axis=1)
    x_test = np.expand_dims(x_test, axis=1)
else:
    x_train = np.transpose(x_train, (0, 3, 1, 2)).copy()
    x_test = np.transpose(x_test, (0, 3, 1, 2)).copy()

x_train = x_train.astype(np.float32)
x_test = x_test.astype(np.float32)

y_train = np.argmax(y_train_oh, axis=1).astype(np.int64)
y_test = np.argmax(y_test_oh, axis=1).astype(np.int64)

print('x_train:', x_train.shape, '| y_train:', y_train.shape)
print('x_test :', x_test.shape,  '| y_test :', y_test.shape)

# (2) Declare model

In [ ]:
class HumanGenderCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # Block 1: (1, 64, 64) -> (16, 32, 32)
        self.conv1_1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv1_2 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv_dropout1 = nn.Dropout(0.2)

        # Block 2: (16, 32, 32) -> (32, 16, 16)
        self.conv2_1 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv2_2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.conv2_3 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv_dropout2 = nn.Dropout(0.2)

        # Block 3: (32, 16, 16) -> (64, 8, 8)
        self.conv3_1 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.conv3_3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.conv_dropout3 = nn.Dropout(0.2)

        # Block 4: (64, 8, 8) -> (128, 4, 4)
        self.conv4_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.conv4_3 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool2d(2, 2)
        self.conv_dropout4 = nn.Dropout(0.2)

        # Block 5: (128, 4, 4) -> (256, 2, 2)
        self.conv5_1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv5_2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv5_3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.pool5 = nn.MaxPool2d(2, 2)
        self.conv_dropout5 = nn.Dropout(0.2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(256 * 2 * 2, 1024)
        self.fc_dropout1 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(1024, 1024)
        self.fc_dropout2 = nn.Dropout(0.2)
        self.fc3 = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1_1(x))
        x = F.relu(self.conv1_2(x))
        x = self.conv_dropout1(self.pool1(x))

        x = F.relu(self.conv2_1(x))
        x = F.relu(self.conv2_2(x))
        x = F.relu(self.conv2_3(x))
        x = self.conv_dropout2(self.pool2(x))

        x = F.relu(self.conv3_1(x))
        x = F.relu(self.conv3_2(x))
        x = F.relu(self.conv3_3(x))
        x = self.conv_dropout3(self.pool3(x))

        x = F.relu(self.conv4_1(x))
        x = F.relu(self.conv4_2(x))
        x = F.relu(self.conv4_3(x))
        x = self.conv_dropout4(self.pool4(x))

        x = F.relu(self.conv5_1(x))
        x = F.relu(self.conv5_2(x))
        x = F.relu(self.conv5_3(x))
        x = self.conv_dropout5(self.pool5(x))

        x = self.flatten(x)
        x = self.fc_dropout1(F.relu(self.fc1(x)))
        x = self.fc_dropout2(F.relu(self.fc2(x)))
        return self.fc3(x)

model = HumanGenderCNN(num_classes=num_classes).to(device)
print(model)
print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')  # Khớp chính xác 4,063,570 params

In [ ]:
try:
    from torchinfo import summary
    print(summary(model, input_size=(1, 1, 64, 64), device=device))
except ImportError:
    print('Cài đặt torchinfo nếu muốn xem bảng summary chi tiết: pip install torchinfo')

# (3) Train defined model (with Checkpoint, EarlyStopping, ReduceLROnPlateau & TensorBoard)

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            logits = model(bx)
            loss = criterion(logits, by)
            total_loss += loss.item() * bx.size(0)
            correct += (logits.argmax(dim=1) == by).sum().item()
            total += bx.size(0)
    return total_loss / total, correct / total


# Hyperparameters giống hệt notebook Keras gốc
lr = 0.001
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

batch_size = 128
epochs = 100000  # Dùng EarlyStopping nên đặt số epoch tối đa rất lớn

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
    batch_size=batch_size, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test)),
    batch_size=batch_size, shuffle=False
)

# ===============================
#      Training utilities
# ===============================
# 1. ModelCheckpoint (save_best_only=True theo val_loss)
ckpt_dir = './checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)
best_val_loss = float('inf')
best_ckpt_path = None

# 2. EarlyStopping (patience=5 theo val_loss)
early_stopping_after_epochs = 5
no_improve_epochs = 0

# 3. ReduceLROnPlateau (factor=0.1, patience=5, min_lr=0.0001)
reduce_lr_after_epochs = 5
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=reduce_lr_after_epochs, min_lr=0.0001
)

# 4. TensorBoard (Chạy: tensorboard --logdir=./logs)
writer = SummaryWriter(log_dir='./logs') if HAS_TENSORBOARD else None

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

starting_time = time.time()
for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * bx.size(0)
        correct += (logits.argmax(dim=1) == by).sum().item()
        total += bx.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
    current_lr = optimizer.param_groups[0]['lr']

    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)

    if writer is not None:
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        writer.add_scalar('Accuracy/train', train_acc, epoch)
        writer.add_scalar('Accuracy/val', val_acc, epoch)
        writer.add_scalar('LearningRate', current_lr, epoch)

    print(f'Epoch {epoch:03d} - lr: {current_lr:.5f} - loss: {train_loss:.4f} - acc: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}')

    # ModelCheckpoint & EarlyStopping logic
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve_epochs = 0
        ckpt_name = f'{epoch:06d}-{train_loss:.4f}-{train_acc:.4f}-{val_loss:.4f}-{val_acc:.4f}.pt'
        best_ckpt_path = os.path.join(ckpt_dir, ckpt_name)
        torch.save(model.state_dict(), best_ckpt_path)
        print(f'  -> Saved best checkpoint: {best_ckpt_path}')
    else:
        no_improve_epochs += 1
        if no_improve_epochs >= early_stopping_after_epochs:
            print(f'EarlyStopping triggered at epoch {epoch} (no val_loss improvement for {early_stopping_after_epochs} epochs).')
            break

    # ReduceLROnPlateau step
    scheduler.step(val_loss)

if writer is not None:
    writer.close()

print('> training time is %.4f minutes' % ((time.time() - starting_time) / 60))

### Load best checkpoint and evaluate again on validation set

In [ ]:
# Tự động tải checkpoint tốt nhất vừa lưu ở bước huấn luyện trên
print('Loading best checkpoint:', best_ckpt_path)
model.load_state_dict(torch.load(best_ckpt_path, map_location=device))
val_loss, val_acc = evaluate(model, test_loader, criterion, device)
print([val_loss, val_acc])

### Plot training curves (bổ sung trực quan hóa trực tiếp ngoài TensorBoard)

In [ ]:
epochs_range = range(1, len(history['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history['loss'], label='train_loss')
axes[0].plot(epochs_range, history['val_loss'], label='val_loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_range, history['accuracy'], label='train_acc')
axes[1].plot(epochs_range, history['val_accuracy'], label='val_acc')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

# (4) Evaluate trained model

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print('Test loss:', test_loss)
print('Test accuracy:', test_acc)

In [ ]:
def int2gender(num):
    return 'female' if num == 0 else 'male'

In [ ]:
# select a test image randomly
random_test_index = np.random.choice(x_test.shape[0], size=1)[0]
test_img = x_test[random_test_index]   # (1, 64, 64)
test_label = int(y_test[random_test_index])

model.eval()
with torch.no_grad():
    inp = torch.from_numpy(np.expand_dims(test_img, axis=0)).to(device)
    pred_label = int(model(inp).argmax(dim=1).item())

plt.imshow(test_img[0], cmap='gray')
plt.title('true label = %s, predicted label = %s' % (int2gender(test_label), int2gender(pred_label)))
plt.show()